# Libraries

In [1]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.19.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

!pip install -q llmcompressor==0.10.0.2
!pip install -q compressed-tensors==0.14.0.1

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 100.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 57.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

2026-05-21 23:38:55.530697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779406735.716962      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779406735.769391      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779406736.167816      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779406736.167862      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779406736.167864      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Phi-4-mini-instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Phi-4-mini-instruct-AWQ-4bit"
TOKENIZER_ID = "microsoft/Phi-4-mini-instruct"

MODEL_NAME           = "Phi-4-mini-instruct"
MODEL_NAME_IN_REPO   = "Phi-4-mini-instruct-AWQ"
COMPRESSION_METHOD   = "AWQ"
MODEL_TYPE           = "Quantization"
#SPARSITY = 70
#PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [7]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Download Model

In [8]:
"""local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)"""

'local_path = snapshot_download(\n    repo_id=MODEL_ID,\n    allow_patterns=f"{PATH}/*",\n    local_dir="/kaggle/working/model"\n)'

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [9]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [10]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [11]:
llm = LLM(
    model=MODEL_ID,
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-21 23:39:24 [utils.py:233] non-default args: {'tokenizer': 'microsoft/Phi-4-mini-instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': 'EdgeCompress01/Phi-4-mini-instruct-AWQ-4bit'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-21 23:39:24 [config.py:446] Replacing legacy 'type' key with 'rope_type'
INFO 05-21 23:39:49 [model.py:549] Resolved architecture: Phi3ForCausalLM
WARNING 05-21 23:39:49 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-21 23:39:49 [model.py:1678] Using max model len 256
INFO 05-21 23:39:50 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-21 23:39:50 [vllm.py:790] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

WARNING 05-21 23:39:52 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

(Worker pid=274) INFO 05-21 23:40:35 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=274) WARNING 05-21 23:40:36 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=275) WARNING 05-21 23:40:36 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=275) WARNING 05-21 23:40:36 [custom_all_reduce.py:165] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=274) WARNING 05-21 23:40:36 [custom_all_reduce.py:165] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=274) INFO 05-21 23:40:36 [parallel_state.py:1716] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP ra

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:06<00:00,  6.52s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:06<00:00,  6.52s/it]
(Worker_TP0 pid=274) 


(Worker_TP0 pid=274) INFO 05-21 23:41:04 [default_loader.py:384] Loading weights took 6.61 seconds
(Worker_TP0 pid=274) INFO 05-21 23:41:05 [gpu_model_runner.py:4820] Model loading took 1.37 GiB memory and 27.700831 seconds
(Worker_TP0 pid=274) INFO 05-21 23:41:19 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/505a659aff/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=274) INFO 05-21 23:41:19 [backends.py:1111] Dynamo bytecode transform time: 13.01 s
(Worker_TP0 pid=274) INFO 05-21 23:41:26 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=274) INFO 05-21 23:41:33 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 12.62 s
(Worker_TP0 pid=274) INFO 05-21 23:41:36 [decorators.py:640] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/b237e7eb599f90ad00782e536906cd64ac3b41aeceab1f05fb996dd164dc0fb1/rank_0_0/model
(Worker_TP0 pid=274) INFO 05-21 23:41:

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:05<00:00,  9.01it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:07<00:00,  4.71it/s]


(Worker_TP1 pid=275) INFO 05-21 23:42:11 [gpu_worker.py:597] CUDA graph pool memory: 1.11 GiB (actual), 1.19 GiB (estimated), difference: 0.08 GiB (7.0%).
(Worker_TP0 pid=274) INFO 05-21 23:42:11 [gpu_model_runner.py:6046] Graph capturing finished in 14 secs, took 1.11 GiB
(Worker_TP0 pid=274) INFO 05-21 23:42:11 [gpu_worker.py:597] CUDA graph pool memory: 1.11 GiB (actual), 1.19 GiB (estimated), difference: 0.08 GiB (7.0%).
(EngineCore pid=249) INFO 05-21 23:42:11 [core.py:283] init engine (profile, create kv cache, warmup model) took 65.37 seconds
(EngineCore pid=249) INFO 05-21 23:42:13 [vllm.py:790] Asynchronous scheduling is enabled.
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times    = []
latency_times = []

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    # Extract from RequestStateStats fields
    ttft    = metrics.first_token_latency                    # prefill cost
    latency = metrics.last_token_ts - metrics.queued_ts      # total e2e latency

    ttft_times.append(ttft)
    latency_times.append(latency)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr,     latency_arr    = np.array(ttft_times), np.array(latency_times)

ttft_mean,    ttft_std       = ttft_arr.mean(),    ttft_arr.std()
latency_mean, latency_std    = latency_arr.mean(), latency_arr.std()

# decode throughput: excludes first token, over decode-only window
throughput_arr               = (MAX_GENERATION_TOKENS - 1) / (latency_arr - ttft_arr)
throughput_mean, throughput_std = throughput_arr.mean(), throughput_arr.std()

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-21 23:42:25 [loggers.py:259] Engine 000: Avg prompt throughput: 15.5 tokens/s, Avg generation throughput: 83.6 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-21 23:42:35 [loggers.py:259] Engine 000: Avg prompt throughput: 15.2 tokens/s, Avg generation throughput: 107.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-21 23:42:45 [loggers.py:259] Engine 000: Avg prompt throughput: 13.3 tokens/s, Avg generation throughput: 106.7 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-21 23:42:55 [loggers.py:259] Engine 000: Avg prompt throughput: 13.3 tokens/s, Avg generation throughput: 105.7 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [13]:
print(latency_times)

[1.3831143059996975, 1.379772793000484, 1.3831924289997914, 1.389499211999464, 1.3900723669994477, 1.392176649999783, 1.3886652920000415, 1.3915751880003882, 1.3922821559999647, 1.3938454660001298, 1.3990155320007034, 1.402120052999635, 1.4047658299996328, 1.404967084999953, 1.3996120159999919, 1.4049553649992959, 1.4059449960004713, 1.405104921000202, 1.4059014350004873, 1.4107446690004508, 1.4145343990003312, 1.417612156999894, 1.4180154179994133, 1.4202058330001819, 1.4203213379996669, 1.4203998860002685, 1.4152832769996166, 1.4197799390003638, 1.4222049599993625, 1.4264500960007354]


In [14]:
print(ttft_times)

[0.03150486946105957, 0.028834104537963867, 0.0284273624420166, 0.03397560119628906, 0.03270459175109863, 0.03290510177612305, 0.03077840805053711, 0.03067779541015625, 0.030808687210083008, 0.029746294021606445, 0.033292293548583984, 0.0332341194152832, 0.036844491958618164, 0.03358197212219238, 0.02785181999206543, 0.031540870666503906, 0.03083515167236328, 0.02791142463684082, 0.028273582458496094, 0.034157752990722656, 0.037949323654174805, 0.03634524345397949, 0.03727579116821289, 0.034040212631225586, 0.035227060317993164, 0.03397822380065918, 0.026520967483520508, 0.03206014633178711, 0.028234481811523438, 0.03469586372375488]


# Results

**Writing in json file**

In [15]:
benchmark_results = {
    "model_name"            : MODEL_NAME,
    "model_type"            : MODEL_TYPE,
    "compression_method"    : COMPRESSION_METHOD,
    #"sparsity"              : SPARSITY,
    "input_token_count"     : input_tokens,
    "generated_token_count" : total_generated_tokens,
    "ttft_sec"              : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "latency_sec"           : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "throughput_tokens_per_sec" : f"{throughput_mean:.2f} ± {throughput_std:.2f}",
}


# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [16]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main c8e78b8] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-AWQ
 1 file changed, 10 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-AWQ/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-AWQ'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   01e6030..c8e78b8  main -> main
